# Apartado 1 - Contexto del Proceso de Ingesta

En este primer apartado se establece el objetivo de la fase inicial del ETL: preparar la sesion de Spark, definir un esquema explicito y garantizar que la lectura del dataset se haga con validaciones de estructura y tipos.

Esto reduce errores silenciosos en etapas posteriores del analisis y del modelado.

## Apartado 2 - Inicializacion de SparkSession

En este apartado se crea una sesion de Spark robusta para ejecutar el pipeline tanto en local como en remoto. Tambien se limpia cualquier sesion previa para evitar conflictos y se valida la conectividad con el master cuando corresponde.

In [1]:
# Paso 2 - Inicializacion de SparkSession
import os
import socket
from urllib.parse import urlparse

from pyspark.sql import SparkSession

# Limpiar cualquier sesión anterior detenida
active_session = SparkSession.getActiveSession()
if active_session is not None:
    try:
        active_session.stop()
    except:
        pass

master_url = os.getenv("SPARK_MASTER", "spark://spark-master:7077")
use_remote_master = os.getenv("SPARK_USE_REMOTE_MASTER", "").strip().lower() in {"1", "true", "yes"}


def master_is_reachable(master: str) -> bool:
    parsed = urlparse(master)
    if parsed.scheme != "spark" or not parsed.hostname or not parsed.port:
        return False
    try:
        with socket.create_connection((parsed.hostname, parsed.port), timeout=2):
            return True
    except OSError:
        return False


spark_master = master_url if use_remote_master and master_is_reachable(master_url) else "local[*]"
if spark_master == "local[*]":
    print("Using local[*] so the smoke test does not depend on remote worker Python versions.")
else:
    print(f"Using remote Spark master: {spark_master}")

spark = (SparkSession.builder
         .appName("IABurnout-Spark-Smoke-Test")
         .master(spark_master)
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.sparkContext.master}")

Using local[*] so the smoke test does not depend on remote worker Python versions.
Spark version: 4.1.1
Spark master: local[*]


## Apartado 3 - Definicion del Esquema de Datos

En este apartado se declara el esquema esperado del dataset con tipos explicitos por columna. Esto permite controlar la calidad de la ingesta y asegura consistencia para transformaciones, feature engineering y entrenamiento futuro de modelos.

In [2]:
# Paso 3 - Definicion del Esquema Estructurado
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

schema = StructType([
    StructField("record_id", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("country", StringType(), True),
    StructField("industry", StringType(), True),
    StructField("job_role", StringType(), True),
    StructField("employment_type", StringType(), True),
    StructField("work_model", StringType(), True),
    StructField("company_size", StringType(), True),
    StructField("age_group", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("years_of_experience", IntegerType(), True),
    StructField("annual_salary_usd", IntegerType(), True),
    StructField("weekly_work_hours", IntegerType(), True),
    StructField("weekly_overtime_hours", IntegerType(), True),
    StructField("mental_health_condition", StringType(), True),
    StructField("has_diagnosis", StringType(), True),
    StructField("treatment_type", StringType(), True),
    StructField("stress_level", StringType(), True),
    StructField("burnout_risk_score", DoubleType(), True),
    StructField("work_life_balance_score", DoubleType(), True),
    StructField("productivity_score", DoubleType(), True),
    StructField("job_satisfaction_score", DoubleType(), True),
    StructField("absenteeism_days_per_year", IntegerType(), True),
    StructField("employer_support_level", StringType(), True),
    StructField("mental_health_policy_exists", StringType(), True),
    StructField("eap_available", StringType(), True),
    StructField("used_eap", StringType(), True),
    StructField("workplace_stigma_felt", StringType(), True),
    StructField("manager_support_score", DoubleType(), True),
    StructField("team_collaboration_score", DoubleType(), True),
    StructField("intention_to_leave", StringType(), True),
    StructField("remote_work_preference", StringType(), True),
    StructField("exercise_days_per_week", IntegerType(), True),
    StructField("sleep_hours_per_night", DoubleType(), True),
])

print(f"Esquema definido con {len(schema.fields)} columnas")

Esquema definido con 34 columnas


In [3]:
from pathlib import Path
import os

# Función para buscar recursivamente hacia arriba hasta encontrar la carpeta 'data'
def find_data_file(filename="mental_health_workplace.csv", max_depth=5):
    """Busca el archivo CSV subiendo en el árbol de directorios."""
    current = Path.cwd()
    for _ in range(max_depth):
        candidate = current / "data" / filename
        if candidate.exists():
            return candidate
        current = current.parent
    return None

# Intenta 1: Búsqueda recursiva desde el directorio actual
dataset_path = find_data_file()

# Intenta 2: Si está en un contenedor Docker, busca en rutas comunes de montaje
if dataset_path is None:
    for docker_mount in ["/work", "/workspace", "/app", "/home/jovyan/work"]:
        candidate = Path(docker_mount) / "data" / "mental_health_workplace.csv"
        if candidate.exists():
            dataset_path = candidate
            break

# Intenta 3: Si SPARK_HOME está configurado, búsqueda desde ahí
if dataset_path is None:
    spark_home = os.getenv("SPARK_HOME")
    if spark_home:
        current = Path(spark_home)
        for _ in range(5):
            candidate = current / "data" / "mental_health_workplace.csv"
            if candidate.exists():
                dataset_path = candidate
                break
            current = current.parent

if dataset_path is None:
    # Debug: mostrar donde estamos buscando
    print(f"❌ No se encontró el archivo!")
    print(f"Directorio de trabajo actual: {Path.cwd()}")
    print(f"SPARK_HOME: {os.getenv('SPARK_HOME', 'No configurado')}")
    print(f"\nIntentando listar directorios:")
    for d in [Path.cwd(), Path.cwd().parent, Path("/work"), Path("/workspace")]:
        if d.exists():
            print(f"\n  {d}:")
            try:
                items = sorted(d.iterdir())[:10]
                for item in items:
                    print(f"    - {item.name}{'/' if item.is_dir() else ''}")
            except PermissionError:
                print(f"    (sin permisos)")
    raise FileNotFoundError(
        "No se encontro mental_health_workplace.csv. "
        "Verifica que el archivo esté en data/ dentro del proyecto."
    )

dataset_path = str(dataset_path)
print(f"✓ Archivo encontrado: {dataset_path}")

df_burnout = (
    spark.read
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(schema)
    .csv(dataset_path)
)

expected_schema = [(f.name, f.dataType.simpleString()) for f in schema.fields]
actual_schema = [(f.name, f.dataType.simpleString()) for f in df_burnout.schema.fields]

if expected_schema != actual_schema:
    actual_schema_dict = dict(actual_schema)
    expected_schema_dict = dict(expected_schema)
    missing = [c for c, _ in expected_schema if c not in actual_schema_dict]
    unexpected = [c for c, _ in actual_schema if c not in expected_schema_dict]
    type_mismatch = [
        (name, exp_t, actual_schema_dict.get(name))
        for name, exp_t in expected_schema
        if name in actual_schema_dict and actual_schema_dict[name] != exp_t
    ]
    raise ValueError(
        "Esquema invalido tras ingesta. "
        f"Faltantes: {missing}. "
        f"Sobrantes: {unexpected}. "
        f"Tipos distintos: {type_mismatch}"
    )

print(f"Dataset cargado correctamente desde: {dataset_path}")
print(f"Filas: {df_burnout.count()} | Columnas: {len(df_burnout.columns)}")
df_burnout.printSchema()
df_burnout.show(5, truncate=False)

✓ Archivo encontrado: /data/mental_health_workplace.csv
Dataset cargado correctamente desde: /data/mental_health_workplace.csv
Filas: 10000 | Columnas: 34
root
 |-- record_id: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- job_role: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- work_model: string (nullable = true)
 |-- company_size: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- years_of_experience: integer (nullable = true)
 |-- annual_salary_usd: integer (nullable = true)
 |-- weekly_work_hours: integer (nullable = true)
 |-- weekly_overtime_hours: integer (nullable = true)
 |-- mental_health_condition: string (nullable = true)
 |-- has_diagnosis: string (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- stress_level: string (nullable = true)
 |-- burnout_risk_score: double (nu

## ETL 1 - Analisis de Datos Faltantes

Calculamos cantidad y porcentaje de valores nulos por columna para priorizar limpieza y decisiones de modelado.

In [4]:
from pyspark.sql import functions as F

row_count = df_burnout.count()

missing_stats = (
    df_burnout
    .select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_burnout.columns
    ])
)

missing_long = (
    missing_stats
    .select(
        F.explode(
            F.array([
                F.struct(
                    F.lit(c).alias("column"),
                    F.col(c).cast("long").alias("missing_count")
                )
                for c in df_burnout.columns
            ])
        ).alias("kv")
    )
    .select(
        F.col("kv.column").alias("column"),
        F.col("kv.missing_count").alias("missing_count")
    )
    .withColumn(
        "missing_pct",
        F.when(F.lit(row_count) > 0, F.round((F.col("missing_count") / F.lit(row_count)) * 100, 2))
         .otherwise(F.lit(0.0))
    )
    .orderBy(F.desc("missing_pct"), F.desc("missing_count"))
)

print(f"Total de filas evaluadas: {row_count}")
missing_long.show(len(df_burnout.columns), truncate=False)

Total de filas evaluadas: 10000
+---------------------------+-------------+-----------+
|column                     |missing_count|missing_pct|
+---------------------------+-------------+-----------+
|record_id                  |0            |0.0        |
|year                       |0            |0.0        |
|country                    |0            |0.0        |
|industry                   |0            |0.0        |
|job_role                   |0            |0.0        |
|employment_type            |0            |0.0        |
|work_model                 |0            |0.0        |
|company_size               |0            |0.0        |
|age_group                  |0            |0.0        |
|gender                     |0            |0.0        |
|years_of_experience        |0            |0.0        |
|annual_salary_usd          |0            |0.0        |
|weekly_work_hours          |0            |0.0        |
|weekly_overtime_hours      |0            |0.0        |
|mental_health_c

## ETL 2 - Limpieza de Registros Incompletos

Eliminamos filas con al menos un valor nulo para quedarnos con registros completos.

In [5]:
rows_before = df_burnout.count()

# Drop de cualquier fila con null en alguna columna
# Si quieres una regla menos estricta, luego podemos usar un subconjunto de columnas criticas.
df_clean = df_burnout.dropna(how="any")

rows_after = df_clean.count()
rows_removed = rows_before - rows_after

print(f"Filas antes de limpieza: {rows_before}")
print(f"Filas despues de limpieza: {rows_after}")
print(f"Filas eliminadas por datos faltantes: {rows_removed}")

Filas antes de limpieza: 10000
Filas despues de limpieza: 10000
Filas eliminadas por datos faltantes: 0


## ETL 3 - Parseo y Normalizacion de Tipos

Estandarizamos columnas de texto y reforzamos cast explicito en variables numericas.

In [6]:
from pyspark.sql.types import IntegerType, DoubleType, StringType

numeric_int_cols = [
    "year", "years_of_experience", "annual_salary_usd", "weekly_work_hours",
    "weekly_overtime_hours", "absenteeism_days_per_year", "exercise_days_per_week"
]

numeric_double_cols = [
    "burnout_risk_score", "work_life_balance_score", "productivity_score",
    "job_satisfaction_score", "manager_support_score",
    "team_collaboration_score", "sleep_hours_per_night"
]

string_cols = [f.name for f in df_clean.schema.fields if isinstance(f.dataType, StringType)]

df_typed = df_clean

for c in numeric_int_cols:
    df_typed = df_typed.withColumn(c, F.col(c).cast(IntegerType()))

for c in numeric_double_cols:
    df_typed = df_typed.withColumn(c, F.col(c).cast(DoubleType()))

for c in string_cols:
    df_typed = df_typed.withColumn(c, F.trim(F.lower(F.col(c))))

print("Esquema despues de parseo/normalizacion:")
df_typed.printSchema()

Esquema despues de parseo/normalizacion:
root
 |-- record_id: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- job_role: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- work_model: string (nullable = true)
 |-- company_size: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- years_of_experience: integer (nullable = true)
 |-- annual_salary_usd: integer (nullable = true)
 |-- weekly_work_hours: integer (nullable = true)
 |-- weekly_overtime_hours: integer (nullable = true)
 |-- mental_health_condition: string (nullable = true)
 |-- has_diagnosis: string (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- stress_level: string (nullable = true)
 |-- burnout_risk_score: double (nullable = true)
 |-- work_life_balance_score: double (nullable = true)
 |-- productivity_score: double (nullable = 

## ETL 4 - Feature Engineering

Construimos variables derivadas para capturar intensidad de carga laboral, compensacion y factores de soporte.

In [7]:
df_features = (
    df_typed
    .withColumn(
        "total_weekly_load",
        F.col("weekly_work_hours") + F.col("weekly_overtime_hours")
    )
    .withColumn(
        "overtime_ratio",
        F.when(F.col("weekly_work_hours") > 0,
               F.round(F.col("weekly_overtime_hours") / F.col("weekly_work_hours"), 4))
         .otherwise(F.lit(0.0))
    )
    .withColumn(
        "salary_per_hour_est",
        F.when(F.col("weekly_work_hours") > 0,
               F.round(F.col("annual_salary_usd") / (F.col("weekly_work_hours") * F.lit(52.0)), 2))
         .otherwise(F.lit(None).cast(DoubleType()))
    )
    .withColumn(
        "is_remote_preference",
        F.when(F.col("remote_work_preference").isin("yes", "si", "true", "1"), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "has_mental_health_diagnosis",
        F.when(F.col("has_diagnosis").isin("yes", "si", "true", "1"), F.lit(1)).otherwise(F.lit(0))
    )
)

print("Vista rapida de features nuevas:")
df_features.select(
    "record_id", "total_weekly_load", "overtime_ratio", "salary_per_hour_est",
    "is_remote_preference", "has_mental_health_diagnosis"
).show(5, truncate=False)

Vista rapida de features nuevas:
+----------+-----------------+--------------+-------------------+--------------------+---------------------------+
|record_id |total_weekly_load|overtime_ratio|salary_per_hour_est|is_remote_preference|has_mental_health_diagnosis|
+----------+-----------------+--------------+-------------------+--------------------+---------------------------+
|mhw0000001|40               |0.0           |30.43              |0                   |1                          |
|mhw0000002|38               |0.0           |32.6               |0                   |1                          |
|mhw0000003|40               |0.0           |60.2               |0                   |1                          |
|mhw0000004|38               |0.0           |23.41              |0                   |0                          |
|mhw0000005|66               |0.2453        |34.54              |0                   |1                          |
+----------+-----------------+--------------+--

## ETL 5 - Seleccion de Columnas para Modelado Futuro

Definimos una vista limpia de variables predictoras y la variable objetivo para un pipeline de ML posterior.

In [8]:
target_col = "burnout_risk_score"

model_feature_cols = [
    "year",
    "industry",
    "job_role",
    "employment_type",
    "work_model",
    "company_size",
    "age_group",
    "gender",
    "years_of_experience",
    "annual_salary_usd",
    "weekly_work_hours",
    "weekly_overtime_hours",
    "stress_level",
    "work_life_balance_score",
    "productivity_score",
    "job_satisfaction_score",
    "absenteeism_days_per_year",
    "employer_support_level",
    "mental_health_policy_exists",
    "eap_available",
    "workplace_stigma_felt",
    "manager_support_score",
    "team_collaboration_score",
    "exercise_days_per_week",
    "sleep_hours_per_night",
    "total_weekly_load",
    "overtime_ratio",
    "salary_per_hour_est",
    "is_remote_preference",
    "has_mental_health_diagnosis"
]

selected_cols = ["record_id"] + model_feature_cols + [target_col]

df_model = df_features.select(*selected_cols).dropna(how="any")

print(f"Columnas seleccionadas para modelado: {len(selected_cols)}")
print(f"Filas finales en df_model: {df_model.count()}")
df_model.printSchema()
df_model.show(5, truncate=False)

Columnas seleccionadas para modelado: 32
Filas finales en df_model: 10000
root
 |-- record_id: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- industry: string (nullable = true)
 |-- job_role: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- work_model: string (nullable = true)
 |-- company_size: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- years_of_experience: integer (nullable = true)
 |-- annual_salary_usd: integer (nullable = true)
 |-- weekly_work_hours: integer (nullable = true)
 |-- weekly_overtime_hours: integer (nullable = true)
 |-- stress_level: string (nullable = true)
 |-- work_life_balance_score: double (nullable = true)
 |-- productivity_score: double (nullable = true)
 |-- job_satisfaction_score: double (nullable = true)
 |-- absenteeism_days_per_year: integer (nullable = true)
 |-- employer_support_level: string (nullable = true)
 |-- mental_health_policy_exis

## ETL 6 - Guardado del Dataset Final en Parquet Particionado

Se persiste `df_model` en formato Parquet para consumo analitico/modelado posterior, particionando por columnas de utilidad para lecturas selectivas.

In [9]:
from pathlib import Path

output_parquet_path = str((Path.cwd() / "output" / "burnout_model_parquet").resolve())

(
    df_model
    .write
    .mode("overwrite")
    .partitionBy("year", "industry")
    .parquet(output_parquet_path)
)

print(f"Dataset final guardado en: {output_parquet_path}")
print("Particiones usadas: year, industry")

Dataset final guardado en: /home/jovyan/work/notebooks/output/burnout_model_parquet
Particiones usadas: year, industry


## ETL 7 - Verificacion de Persistencia Parquet

Se recarga el Parquet generado, se valida conteo/esquema y se inspecciona una muestra para confirmar que el guardado funciona correctamente.

In [10]:
df_model_check = spark.read.parquet(output_parquet_path)

print(f"Filas verificadas desde parquet: {df_model_check.count()}")
print(f"Columnas verificadas desde parquet: {len(df_model_check.columns)}")

print("Esquema recargado:")
df_model_check.printSchema()

print("Muestra de datos recargados:")
df_model_check.show(5, truncate=False)

print(f"Numero de archivos leidos: {len(df_model_check.inputFiles())}")

Filas verificadas desde parquet: 10000
Columnas verificadas desde parquet: 32
Esquema recargado:
root
 |-- record_id: string (nullable = true)
 |-- job_role: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- work_model: string (nullable = true)
 |-- company_size: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- years_of_experience: integer (nullable = true)
 |-- annual_salary_usd: integer (nullable = true)
 |-- weekly_work_hours: integer (nullable = true)
 |-- weekly_overtime_hours: integer (nullable = true)
 |-- stress_level: string (nullable = true)
 |-- work_life_balance_score: double (nullable = true)
 |-- productivity_score: double (nullable = true)
 |-- job_satisfaction_score: double (nullable = true)
 |-- absenteeism_days_per_year: integer (nullable = true)
 |-- employer_support_level: string (nullable = true)
 |-- mental_health_policy_exists: string (nullable = true)
 |-- eap_available: strin